In [2]:
"""
================================================================================
Experiment 54: Energy Spectral Bandwidth MFCC + CNN + XGBoost + Class Weights
================================================================================
This script performs patient-level 3-fold cross-validation for:
- Feature extraction: Energy (RMS) + Spectral Bandwidth + MFCC (13 coefficients) -> 2D (15, time)
- Method: CNN features (1D CNN) + XGBoost classifier with class weights
- Minor classes (Healthy, URTI) are merged into 'Other' for stable stratification.
- Calculates per-class metrics (precision, recall, f1, fpr, fnr) for each fold
  and aggregates them across folds.
- Uses class weights for imbalance handling (no SMOTE).

All comments and output messages are in English for GitHub sharing.

Author: Ehab Hassan Abdel Hady
Date: 2025
Based on original code from the paper.
"""

import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    balanced_accuracy_score, confusion_matrix,
    precision_recall_fscore_support
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, GlobalAveragePooling1D, Input
import xgboost as xgb
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import json
import pickle
import time
from datetime import timedelta
from joblib import Parallel, delayed
from tqdm import tqdm

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# =============================================================================
# Configuration – EDIT THESE PATHS
# =============================================================================
SOUND_FILES_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\audio_and_txt_files\\"
DIAGNOSIS_CSV_PATH = "E:\\FFF\\respiratory-sound-database\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\patient_diagnosis.csv"
OUTPUT_DIR = "E:\\Results\\exp54_classweight\\"          # change per experiment

# Number of parallel jobs for feature extraction (adjust to your CPU cores)
N_JOBS = 8

# =============================================================================
# Internal settings (do not change unless you know what you are doing)
# =============================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURES_RAW_FILE = os.path.join(OUTPUT_DIR, "features_raw.npy")
LABELS_RAW_FILE = os.path.join(OUTPUT_DIR, "labels_raw.npy")
PATIENT_IDS_RAW_FILE = os.path.join(OUTPUT_DIR, "patient_ids_raw.npy")
CLEANED_DATA_FILE = os.path.join(OUTPUT_DIR, "cleaned_data.pkl")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
RESULTS_FILE = os.path.join(OUTPUT_DIR, "fold_results.csv")
PER_CLASS_RESULTS_FILE = os.path.join(OUTPUT_DIR, "per_class_results.csv")
TIMING_FILE = os.path.join(OUTPUT_DIR, "timing_log.txt")

WEIGHTS = {
    'macro_f1': 0.249,
    'macro_recall': 0.249,
    'weighted_f1': 0.157,
    'weighted_recall': 0.096,
    'accuracy': 0.096,
    'balanced_accuracy': 0.058,
    'inverted_fpr': 0.058,
    'fnr': 0.037
}

# =============================================================================
# Utility functions
# =============================================================================
def save_progress(stage, fold=None):
    progress = {'stage': stage}
    if fold is not None:
        progress['last_completed_fold'] = fold
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f)

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            progress = json.load(f)
        return progress.get('stage', 1), progress.get('last_completed_fold', 0)
    return 1, 0

def log_time(stage_name, elapsed_seconds):
    line = f"{stage_name}: {timedelta(seconds=elapsed_seconds)} (hh:mm:ss)\n"
    with open(TIMING_FILE, 'a') as f:
        f.write(line)
    print(line.strip())

# =============================================================================
# Feature extraction (Energy + Spectral Bandwidth + MFCC)
# =============================================================================
def extract_features_single(file_path, max_pad_len=862):
    """
    Load audio file and extract:
    - Energy (RMS) : 1 row
    - Spectral Bandwidth : 1 row
    - MFCC (13 coefficients) : 13 rows
    Returns a 2D array of shape (15, max_pad_len) or None if error.
    """
    try:
        audio, sr = sf.read(file_path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        target_len = 20 * sr
        if len(audio) > target_len:
            audio = audio[:target_len]
        elif len(audio) < target_len:
            repeats = target_len // len(audio) + 1
            audio = np.tile(audio, repeats)[:target_len]

        # Energy (RMS)
        rms = librosa.feature.rms(y=audio)                           # (1, time)
        # Spectral Bandwidth
        spec_bw = librosa.feature.spectral_bandwidth(y=audio, sr=sr) # (1, time)
        # MFCCs (13 coefficients)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)     # (13, time)

        features = np.vstack([rms, spec_bw, mfccs])                  # (15, time)

        if features.shape[1] < max_pad_len:
            pad_width = max_pad_len - features.shape[1]
            features = np.pad(features, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            features = features[:, :max_pad_len]

        return features
    except Exception:
        return None

# =============================================================================
# Per-class metrics
# =============================================================================
def per_class_metrics(y_true, y_pred, class_names):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    FP = cm.sum(axis=0) - np.diag(cm)
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    with np.errstate(divide='ignore', invalid='ignore'):
        FPR = np.divide(FP, FP + TN, where=(FP+TN) > 0)
        FNR = np.divide(FN, FN + TP, where=(FN+TP) > 0)
        FPR = np.nan_to_num(FPR, nan=0.0)
        FNR = np.nan_to_num(FNR, nan=0.0)

    result = {}
    for i, name in enumerate(class_names):
        result[name] = {
            'precision': precision[i],
            'recall': recall[i],
            'f1': f1[i],
            'fpr': FPR[i],
            'fnr': FNR[i]
        }
    return result

# =============================================================================
# Main execution with checkpointing
# =============================================================================
print("=" * 60)
print("Experiment 54 – Energy Spectral Bandwidth MFCC + CNN + XGBoost + Class Weights")
print("=" * 60)

current_stage, last_fold = load_progress()
print(f"Resuming from stage {current_stage}, last completed fold: {last_fold}")

# -----------------------------------------------------------------------------
# STAGE 1: Extract raw features
# -----------------------------------------------------------------------------
if current_stage == 1:
    stage_start = time.time()
    print("\n[Stage 1] Extracting features from audio files...")

    filenames = [f for f in os.listdir(SOUND_FILES_PATH)
                 if f.endswith('.wav') and os.path.isfile(os.path.join(SOUND_FILES_PATH, f))]
    print(f"    Found {len(filenames)} audio files.")
    filepaths = [os.path.join(SOUND_FILES_PATH, f) for f in filenames]
    patient_ids = [int(name[:3]) for name in filenames]

    p_diag = pd.read_csv(DIAGNOSIS_CSV_PATH, header=None)
    diagnosis_dict = dict(zip(p_diag[0], p_diag[1]))
    labels = [diagnosis_dict.get(pid, None) for pid in patient_ids]

    print(f"    Using {N_JOBS} parallel workers...")
    results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(extract_features_single)(fp) for fp in tqdm(filepaths, desc="Feature extraction")
    )

    features = []
    valid_indices = []
    for i, feat in enumerate(results):
        if feat is not None:
            features.append(feat)
            valid_indices.append(i)

    errors = len(filepaths) - len(features)
    print(f"    Extraction complete: {len(features)} successful, {errors} errors.")
    if len(features) == 0:
        raise RuntimeError("No features extracted.")

    features = np.array(features)  # (n_samples, 15, 862)
    patient_ids = [patient_ids[i] for i in valid_indices]
    labels = [labels[i] for i in valid_indices]

    np.save(FEATURES_RAW_FILE, features)
    np.save(LABELS_RAW_FILE, labels)
    np.save(PATIENT_IDS_RAW_FILE, patient_ids)
    print(f"    Raw features saved to {FEATURES_RAW_FILE}")

    elapsed = time.time() - stage_start
    log_time("Stage 1 - Feature extraction", elapsed)
    save_progress(2)
    current_stage = 2

# -----------------------------------------------------------------------------
# STAGE 2: Clean and prepare data (remove Asthma/LRTI, merge Healthy/URTI)
# -----------------------------------------------------------------------------
if current_stage == 2:
    stage_start = time.time()
    print("\n[Stage 2] Cleaning and preparing data...")

    features = np.load(FEATURES_RAW_FILE, allow_pickle=True)
    labels = np.load(LABELS_RAW_FILE, allow_pickle=True).tolist()
    patient_ids = np.load(PATIENT_IDS_RAW_FILE, allow_pickle=True).tolist()

    valid = [i for i, lbl in enumerate(labels) if lbl is not None]
    features = features[valid]
    patient_ids = [patient_ids[i] for i in valid]
    labels = [labels[i] for i in valid]

    new_features = []
    new_labels = []
    new_patient_ids = []
    for f, lbl, pid in zip(features, labels, patient_ids):
        if lbl in ['Asthma', 'LRTI']:
            continue
        elif lbl in ['Healthy', 'URTI']:
            new_labels.append('Other')
            new_features.append(f)
            new_patient_ids.append(pid)
        else:
            new_labels.append(lbl)
            new_features.append(f)
            new_patient_ids.append(pid)

    features = np.array(new_features)
    labels = new_labels
    patient_ids = new_patient_ids

    print(f"    Remaining samples after cleaning: {len(features)}")
    print("    New class distribution:")
    print(Counter(labels))

    # Transpose to (samples, time, features) for 1D CNN
    features = features.transpose(0, 2, 1)  # (n_samples, 862, 15)

    cleaned_data = {
        'features': features,
        'labels': labels,
        'patient_ids': patient_ids
    }
    with open(CLEANED_DATA_FILE, 'wb') as f:
        pickle.dump(cleaned_data, f)
    print(f"    Cleaned data saved to {CLEANED_DATA_FILE}")

    elapsed = time.time() - stage_start
    log_time("Stage 2 - Data cleaning", elapsed)
    save_progress(3)
    current_stage = 3

# -----------------------------------------------------------------------------
# STAGE 3: Cross-validation (3 folds) with class weights
# -----------------------------------------------------------------------------
if current_stage == 3:
    stage_start = time.time()
    print("\n[Stage 3] Running cross‑validation...")

    with open(CLEANED_DATA_FILE, 'rb') as f:
        cleaned = pickle.load(f)
    features = cleaned['features']
    labels = cleaned['labels']
    patient_ids = cleaned['patient_ids']

    df = pd.DataFrame({
        'features': list(features),
        'label': labels,
        'patient_id': patient_ids
    })

    unique_patients = df.groupby('patient_id')['label'].first().reset_index()
    le = LabelEncoder()
    unique_patients['label_enc'] = le.fit_transform(unique_patients['label'])

    print(f"    Unique patients: {len(unique_patients)}")
    print(f"    Class distribution among patients:\n{unique_patients['label'].value_counts()}")

    n_splits = 3
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    if os.path.exists(RESULTS_FILE):
        fold_results_df = pd.read_csv(RESULTS_FILE)
        fold_results = fold_results_df.to_dict('records')
        completed_folds = set(fold_results_df['fold'].tolist())
    else:
        fold_results = []
        completed_folds = set()

    if os.path.exists(PER_CLASS_RESULTS_FILE):
        per_class_df = pd.read_csv(PER_CLASS_RESULTS_FILE)
        per_class_fold_data = per_class_df.to_dict('records')
    else:
        per_class_fold_data = []

    for fold, (train_pat_idx, test_pat_idx) in enumerate(skf.split(
            unique_patients['patient_id'], unique_patients['label_enc'])):
        fold_num = fold + 1
        if fold_num in completed_folds:
            print(f"\n--- Fold {fold_num} already completed, skipping ---")
            continue

        fold_start = time.time()
        print(f"\n--- Fold {fold_num}/{n_splits} ---")

        train_patients = unique_patients.iloc[train_pat_idx]['patient_id'].values
        test_patients = unique_patients.iloc[test_pat_idx]['patient_id'].values

        train_df = df[df['patient_id'].isin(train_patients)]
        test_df = df[df['patient_id'].isin(test_patients)]

        X_train = np.stack(train_df['features'].values)
        X_test = np.stack(test_df['features'].values)
        y_train_str = train_df['label'].values
        y_test_str = test_df['label'].values

        le_fold = LabelEncoder()
        y_train = le_fold.fit_transform(y_train_str)
        y_test = le_fold.transform(y_test_str)
        classes = np.unique(y_train)

        print(f"    Train samples: {len(X_train)}, Test samples: {len(X_test)}")
        print(f"    Train patients: {len(train_patients)}, Test patients: {len(test_patients)}")
        print(f"    Classes in train: {le_fold.inverse_transform(classes)}")

        # --- CNN feature extractor (input shape (862, 15)) ---
        cnn = Sequential([
            Input(shape=(862, 15)),
            Conv1D(16, kernel_size=2, activation='relu'),
            MaxPooling1D(pool_size=2),
            Dropout(0.2),
            Conv1D(32, kernel_size=2, activation='relu'),
            MaxPooling1D(pool_size=2),
            Dropout(0.2),
            Conv1D(64, kernel_size=2, activation='relu'),
            MaxPooling1D(pool_size=2),
            Dropout(0.2),
            Conv1D(128, kernel_size=2, activation='relu'),
            MaxPooling1D(pool_size=2),
            Dropout(0.2),
            GlobalAveragePooling1D()
        ])
        cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

        X_train_feat = cnn.predict(X_train, verbose=0)
        X_test_feat = cnn.predict(X_test, verbose=0)
        X_train_feat = X_train_feat.reshape(X_train_feat.shape[0], -1)
        X_test_feat = X_test_feat.reshape(X_test_feat.shape[0], -1)

        # --- Class weights for imbalance handling ---
        class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
        sample_weight = np.array([class_weights[y] for y in y_train])
        print("    Using class weights.")

        # --- XGBoost training with sample weights ---
        xgb_model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.8,
            gamma=0.1,
            use_label_encoder=False,
            eval_metric='mlogloss',
            random_state=42
        )
        xgb_model.fit(X_train_feat, y_train, sample_weight=sample_weight)

        y_pred = xgb_model.predict(X_test_feat)

        # --- metrics calculation ---
        cm = confusion_matrix(y_test, y_pred, labels=classes)
        FP = cm.sum(axis=0) - np.diag(cm)
        FN = cm.sum(axis=1) - np.diag(cm)
        TP = np.diag(cm)
        TN = cm.sum() - (FP + FN + TP)

        with np.errstate(divide='ignore', invalid='ignore'):
            FPR_per_class = np.divide(FP, FP + TN, where=(FP+TN) > 0)
            FNR_per_class = np.divide(FN, FN + TP, where=(FN+TP) > 0)
            FPR_per_class = np.nan_to_num(FPR_per_class, nan=0.0)
            FNR_per_class = np.nan_to_num(FNR_per_class, nan=0.0)

        macro_fpr = np.mean(FPR_per_class)
        macro_fnr = np.mean(FNR_per_class)

        fold_metrics = {
            'fold': fold_num,
            'accuracy': accuracy_score(y_test, y_pred),
            'macro_f1': f1_score(y_test, y_pred, average='macro', zero_division=0),
            'macro_recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
            'weighted_f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'weighted_recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
            'fpr': macro_fpr,
            'fnr': macro_fnr
        }
        fold_results.append(fold_metrics)

        class_names = le_fold.classes_
        per_class = per_class_metrics(y_test, y_pred, class_names)
        for class_name, metrics in per_class.items():
            per_class_fold_data.append({
                'fold': fold_num,
                'class': class_name,
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1'],
                'fpr': metrics['fpr'],
                'fnr': metrics['fnr']
            })

        pd.DataFrame(fold_results).to_csv(RESULTS_FILE, index=False)
        pd.DataFrame(per_class_fold_data).to_csv(PER_CLASS_RESULTS_FILE, index=False)
        save_progress(3, fold_num)

        fold_elapsed = time.time() - fold_start
        print(f"    Fold {fold_num} completed in {timedelta(seconds=fold_elapsed)}.")
        print(f"    Results saved.")

    elapsed = time.time() - stage_start
    log_time("Stage 3 - Cross-validation", elapsed)
    print("\nAll folds completed.")
    save_progress(4)

# -----------------------------------------------------------------------------
# STAGE 4: Aggregate final results
# -----------------------------------------------------------------------------
if current_stage == 4 or (current_stage >= 3 and os.path.exists(RESULTS_FILE)):
    print("\n[Stage 4] Aggregating final results...")
    fold_results_df = pd.read_csv(RESULTS_FILE)
    fold_results = fold_results_df.to_dict('records')

    avg_metrics = {}
    for metric in ['accuracy', 'macro_f1', 'macro_recall', 'weighted_f1',
                   'weighted_recall', 'balanced_accuracy', 'fpr', 'fnr']:
        values = [f[metric] for f in fold_results]
        avg_metrics[metric] = np.mean(values)
        avg_metrics[f'{metric}_std'] = np.std(values)

    inverted_fpr = 1 - avg_metrics['fpr']
    mcdm_score = (
        avg_metrics['macro_f1'] * WEIGHTS['macro_f1'] +
        avg_metrics['macro_recall'] * WEIGHTS['macro_recall'] +
        avg_metrics['weighted_f1'] * WEIGHTS['weighted_f1'] +
        avg_metrics['weighted_recall'] * WEIGHTS['weighted_recall'] +
        avg_metrics['accuracy'] * WEIGHTS['accuracy'] +
        avg_metrics['balanced_accuracy'] * WEIGHTS['balanced_accuracy'] +
        inverted_fpr * WEIGHTS['inverted_fpr'] +
        avg_metrics['fnr'] * WEIGHTS['fnr']
    )

    print("\n" + "=" * 50)
    print("FINAL AVERAGED METRICS (over {} folds)".format(len(fold_results)))
    print("=" * 50)
    for metric in ['accuracy', 'macro_f1', 'macro_recall', 'weighted_f1',
                   'weighted_recall', 'balanced_accuracy', 'fpr', 'fnr']:
        print(f"  {metric:18s}: {avg_metrics[metric]:.4f} ± {avg_metrics[f'{metric}_std']:.4f}")
    print(f"  inverted_fpr        : {inverted_fpr:.4f}")
    print("-" * 40)
    print(f"  *** MCDM SCORE      : {mcdm_score:.4f} ***")
    print("=" * 50)

    summary = {
        'metric': ['accuracy', 'macro_f1', 'macro_recall', 'weighted_f1',
                   'weighted_recall', 'balanced_accuracy', 'fpr', 'fnr', 'mcdm_score'],
        'mean': [avg_metrics['accuracy'], avg_metrics['macro_f1'], avg_metrics['macro_recall'],
                 avg_metrics['weighted_f1'], avg_metrics['weighted_recall'],
                 avg_metrics['balanced_accuracy'], avg_metrics['fpr'], avg_metrics['fnr'], mcdm_score],
        'std': [avg_metrics['accuracy_std'], avg_metrics['macro_f1_std'], avg_metrics['macro_recall_std'],
                avg_metrics['weighted_f1_std'], avg_metrics['weighted_recall_std'],
                avg_metrics['balanced_accuracy_std'], avg_metrics['fpr_std'], avg_metrics['fnr_std'], np.nan]
    }
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "final_summary.csv"), index=False)
    print(f"\nSummary saved to {os.path.join(OUTPUT_DIR, 'final_summary.csv')}")

    if os.path.exists(TIMING_FILE):
        print("\n" + "=" * 50)
        print("TIMING LOG")
        with open(TIMING_FILE, 'r') as f:
            print(f.read())

print("\n" + "=" * 60)
print("Experiment 54 completed successfully.")
print("=" * 60)

Experiment 54 – Energy Spectral Bandwidth MFCC + CNN + XGBoost + Class Weights
Resuming from stage 1, last completed fold: 0

[Stage 1] Extracting features from audio files...
    Found 920 audio files.
    Using 8 parallel workers...


Feature extraction:   2%|▏         | 16/920 [00:03<03:17,  4.58it/s][Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:    3.6s
[Parallel(n_jobs=8)]: Done   9 tasks      | elapsed:    4.3s
Feature extraction: 100%|██████████| 920/920 [01:47<00:00,  8.56it/s]
[Parallel(n_jobs=8)]: Done 920 out of 920 | elapsed:  1.8min finished


    Extraction complete: 920 successful, 0 errors.
    Raw features saved to E:\Results\exp54_classweight\features_raw.npy
Stage 1 - Feature extraction: 0:01:48.695536 (hh:mm:ss)

[Stage 2] Cleaning and preparing data...
    Remaining samples after cleaning: 917
    New class distribution:
Counter({'COPD': 793, 'Other': 58, 'Pneumonia': 37, 'Bronchiectasis': 16, 'Bronchiolitis': 13})
    Cleaned data saved to E:\Results\exp54_classweight\cleaned_data.pkl
Stage 2 - Data cleaning: 0:00:00.345794 (hh:mm:ss)

[Stage 3] Running cross‑validation...
    Unique patients: 123
    Class distribution among patients:
label
COPD              64
Other             40
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
Name: count, dtype: int64

--- Fold 1/3 ---
    Train samples: 655, Test samples: 262
    Train patients: 82, Test patients: 41
    Classes in train: ['Bronchiectasis' 'Bronchiolitis' 'COPD' 'Other' 'Pneumonia']
    Using class weights.
    Fold 1 completed in 0:00:01.420444.